# Day 22 — 资金管理与仓位控制

**核心问题**：有了策略（Day20）和边界条件（Day21），每次该用多少钱买？

**三个关键词**：凯利公式 | 仓位计算方法 | 连续亏损心理止损

## 一、凯利公式 — 理论最优仓位

$$f^* = \frac{p \times b - q}{b}$$

- $p$ = 胜率 | $q = 1-p$ = 败率 | $b$ = 盈亏比（均盈利÷均亏损绝对值）
- 直觉：每次有60%概率赚20%、40%概率亏10% → f* = (0.6×2-0.4)/2 = 40%

**凯利告诉你：盈亏比<1的策略根本不值得下注。**

In [1]:
import pandas as pd
import numpy as np

def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def calc_all_signals(df):
    d = df.copy()
    d['MA5'] = d['收盘'].rolling(5).mean()
    d['MA20'] = d['收盘'].rolling(20).mean()
    d['MA60'] = d['收盘'].rolling(60).mean()
    ema12 = d['收盘'].ewm(span=12, adjust=False).mean()
    ema26 = d['收盘'].ewm(span=26, adjust=False).mean()
    d['DIF'] = ema12 - ema26
    d['DEA'] = d['DIF'].ewm(span=9, adjust=False).mean()
    d['RSI'] = calc_rsi(d['收盘'], 14)
    bb_mid = d['收盘'].rolling(20).mean()
    bb_std = d['收盘'].rolling(20).std()
    d['%B'] = (d['收盘'] - (bb_mid - 2*bb_std)) / (4*bb_std)
    d['量MA20'] = d['成交量'].rolling(20).mean()
    return d

STOCKS = {
    '海康威视': '/tmp/海康威视.csv', '恒瑞医药': '/tmp/恒瑞医药.csv',
    '比亚迪': '/tmp/byd.csv', '茅台': '/tmp/mt.csv', '招商银行': '/tmp/zs.csv',
}

def macd_trades(df, hold_days=10):
    trades = []
    in_position = False
    for i in range(1, len(df)):
        if not in_position:
            if df['DIF'].iloc[i] > df['DEA'].iloc[i] and df['DIF'].iloc[i-1] <= df['DEA'].iloc[i-1]:
                in_position = True
                buy_price = df['收盘'].iloc[i]
        else:
            days_held = i - (i - 1)  # simplified
            ret = df['收盘'].iloc[i] / buy_price - 1
            if (i - df.index.get_loc(df.index[i-1]) + 1) >= hold_days or ret <= -0.05:
                trades.append(ret * 100)
                in_position = False
    return trades

# 计算每只股票的凯利参数
print(f'{"股票":8s} {"笔数":>4s} {"胜率":>6s} {"均盈利":>7s} {"均亏损":>7s} {"盈亏比":>6s} {"凯利f*":>7s}')
print('-' * 55)

for name, path in STOCKS.items():
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]
    
    # 纯MACD金叉回测
    trades = []
    in_pos = False
    buy_price = 0
    for i in range(1, len(d)):
        if not in_pos:
            if d['DIF'].iloc[i] > d['DEA'].iloc[i] and d['DIF'].iloc[i-1] <= d['DEA'].iloc[i-1]:
                in_pos = True
                buy_price = d['收盘'].iloc[i]
                buy_i = i
        else:
            days = i - buy_i
            ret = d['收盘'].iloc[i] / buy_price - 1
            if days >= 10 or ret <= -0.05:
                trades.append(ret * 100)
                in_pos = False
    
    if len(trades) < 2:
        continue
    wins = [t for t in trades if t > 0]
    losses = [t for t in trades if t <= 0]
    p = len(wins) / len(trades)
    avg_win = np.mean(wins) if wins else 0
    avg_loss = abs(np.mean(losses)) if losses else 1
    b = avg_win / avg_loss if avg_loss > 0 else 0
    kelly = (p * b - (1-p)) / b if b > 0 else 0
    print(f'{name:6s} {len(trades):4d} {p:5.0%} {avg_win:+7.2f}% {avg_loss:+7.2f}% {b:6.2f} {kelly:7.1%}')

print('\n⚠️ 四只股票凯利为负 → 策略盈亏比<1，不值得下注')
print('✅ 只有海康(牛市)的MACD策略有正凯利23.7%')

股票         笔数     胜率     均盈利     均亏损    盈亏比    凯利f*
-------------------------------------------------------
海康威视      7   57%   +6.44%   +5.03%   1.28   23.7%
恒瑞医药      8   50%   +3.62%   +6.47%   0.56  -39.4%
比亚迪       5   40%   +2.15%   +4.83%   0.44  -95.0%
茅台        8   25%   +2.35%   +3.16%   0.74  -75.9%
招商银行      6   67%   +1.20%   +4.78%   0.25  -66.0%

⚠️ 四只股票凯利为负 → 策略盈亏比<1，不值得下注
✅ 只有海康(牛市)的MACD策略有正凯利23.7%


## 二、三种仓位计算方法

| 方法 | 公式 | 优点 | 缺点 |
|------|------|------|------|
| 固定比例 | 风险金额÷(买价-止损) | 简单一致 | 忽略波动率 |
| 凯利 | 资金×f* | 理论最优 | 参数不稳定 |
| ATR法 | 风险金额÷(ATR×2) | 自动适应波动 | 需要计算ATR |

In [2]:
# 三种方法仓位计算
capital = 100000  # 总资金10万
risk_pct = 0.02   # 每笔风险2%
risk_amount = capital * risk_pct

print(f'{"股票":8s} {"现价":>7s} {"ATR":>7s} {"固定法":>8s} {"ATR法":>7s} {"说明":>20s}')
print('-' * 60)

# 从前面分析中取凯利值
kelly_map = {'海康威视': 0.118, '恒瑞医药': 0, '比亚迪': 0, '茅台': 0, '招商银行': 0}

for name, path in STOCKS.items():
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index().iloc[-60:]
    price = df['收盘'].iloc[-1]
    
    # ATR
    hl = df['最高'] - df['最低']
    hc = abs(df['最高'] - df['收盘'].shift(1))
    lc = abs(df['最低'] - df['收盘'].shift(1))
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().iloc[-1]
    
    stop_price = price * 0.95
    shares_fixed = risk_amount / (price - stop_price)
    shares_atr = risk_amount / (atr * 2) if atr > 0 else 0
    
    diff_pct = (shares_atr / shares_fixed - 1) * 100 if shares_fixed > 0 else 0
    note = '低波动加仓' if diff_pct > 5 else ('高波动减仓' if diff_pct < -5 else '基本一致')
    print(f'{name:6s} {price:7.2f} {atr:7.2f} {int(shares_fixed):8d}股 {int(shares_atr):7d}股 {note:>20s}')

print('\n💡 ATR法的优雅之处：高波动股票自动减仓，低波动股票自动加仓')

股票            现价     ATR      固定法    ATR法                   说明
------------------------------------------------------------
海康威视     34.45    1.74     1161股     576股                高波动减仓
恒瑞医药     55.75    2.72      717股     368股                高波动减仓
比亚迪      90.00    3.38      444股     296股                高波动减仓
茅台     1204.98   31.24       33股      32股                 基本一致
招商银行     36.88    0.83     1084股    1200股                低波动加仓

💡 ATR法的优雅之处：高波动股票自动减仓，低波动股票自动加仓


## 三、连续亏损 — 仓位越大，坑越深

**核心定律**：跌X%需要涨Y%才能回本
- 跌10% → 涨11.1%回本
- 跌20% → 涨25.0%回本
- 跌50% → **涨100%回本（翻倍！）**

In [3]:
import numpy as np
np.random.seed(42)

def simulate(capital, position_pct, p_win, b_ratio, n_trades, loss_start=0):
    history = [capital]
    max_cap = capital
    max_dd = 0
    for i in range(n_trades):
        win = False if i < loss_start else np.random.random() < p_win
        if win:
            ret = np.random.uniform(0.01, 0.15) * b_ratio
        else:
            ret = -np.random.uniform(0.01, 0.10)
        capital *= (1 + ret * position_pct)
        history.append(capital)
        max_cap = max(max_cap, capital)
        max_dd = min(max_dd, (capital - max_cap) / max_cap * 100)
    return capital, max_dd, history

# 正常情况 vs 连亏5次开局
print('=== 正常20笔交易（胜率50%，盈亏比1.5）===')
print(f'{"仓位":8s} {"最终资金":>10s} {"收益":>8s} {"最大回撤":>8s}')
for pct in [0.10, 0.20, 0.30, 0.50]:
    final, mdd, _ = simulate(100000, pct, 0.50, 1.5, 20)
    print(f'{pct:6.0%} {final:10.0f}元 {(final/100000-1)*100:+7.1f}% {mdd:+8.1f}%')

print('\n=== 连亏5次开局（熊市初期）===')
print(f'{"仓位":8s} {"最终资金":>10s} {"收益":>8s} {"最大回撤":>8s}')
for pct in [0.10, 0.20, 0.30, 0.50]:
    final, mdd, _ = simulate(100000, pct, 0.50, 1.5, 20, loss_start=5)
    print(f'{pct:6.0%} {final:10.0f}元 {(final/100000-1)*100:+7.1f}% {mdd:+8.1f}%')

# 回本需求表
print('\n=== 跌X%需要涨Y%才能回本 ===')
for loss in [5, 10, 20, 30, 50]:
    gain_needed = (100 / (100 - loss) - 1) * 100
    print(f'  跌{loss:2d}% → 需要涨{gain_needed:5.1f}% 回本')
print('\n💀 仓位管理不是让你赚更多，是让你活更久。')

=== 正常20笔交易（胜率50%，盈亏比1.5）===
仓位             最终资金       收益     最大回撤
   10%     112775元   +12.8%     -1.0%
   20%     124493元   +24.5%     -5.7%
   30%     105490元    +5.5%     -7.5%
   50%     130285元   +30.3%     -5.5%

=== 连亏5次开局（熊市初期）===
仓位             最终资金       收益     最大回撤
   10%     103246元    +3.2%     -3.4%
   20%      98436元    -1.6%     -7.7%
   30%      96600元    -3.4%     -9.1%
   50%      81820元   -18.2%    -30.8%

=== 跌X%需要涨Y%才能回本 ===
  跌 5% → 需要涨  5.3% 回本
  跌10% → 需要涨 11.1% 回本
  跌20% → 需要涨 25.0% 回本
  跌30% → 需要涨 42.9% 回本
  跌50% → 需要涨100.0% 回本

💀 仓位管理不是让你赚更多，是让你活更久。


## 四、V3仓位规则 vs 固定30%

V3规则：牛市40% / 震荡30% / 熊市20%

In [4]:
print(f'{"股票":8s} {"环境":4s} {"MA多头%":>7s} {"固定30%":>9s} {"V3仓位":>9s} {"胜者":>6s}')
print('-' * 55)

for name, path in STOCKS.items():
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = calc_all_signals(df).iloc[60:]
    
    ma_pct = (d['MA20'] > d['MA60']).sum() / len(d) * 100
    
    if ma_pct > 50:
        regime, v3_pos = '牛市', 0.40
    elif ma_pct > 30:
        regime, v3_pos = '震荡', 0.30
    else:
        regime, v3_pos = '熊市', 0.20
    
    # MACD交易
    trades = []
    in_pos = False
    for i in range(1, len(d)):
        if not in_pos:
            if d['DIF'].iloc[i] > d['DEA'].iloc[i] and d['DIF'].iloc[i-1] <= d['DEA'].iloc[i-1]:
                in_pos = True
                bp = d['收盘'].iloc[i]
                bi = i
        else:
            if i - bi >= 10 or d['收盘'].iloc[i] / bp - 1 <= -0.05:
                trades.append((d['收盘'].iloc[i] / bp - 1) * 100)
                in_pos = False
    
    if not trades:
        continue
    fixed_ret = sum(trades) * 0.30
    v3_ret = sum(trades) * v3_pos
    winner = 'V3✅' if v3_ret > fixed_ret else ('固定' if fixed_ret > v3_ret else '平')
    print(f'{name:6s} {regime:4s} {ma_pct:6.0f}% {fixed_ret:+8.1f}% {v3_ret:+8.1f}% ({v3_pos:.0%}) {winner:>6s}')

print('\n📊 V3分级仓位在4/5股票上跑赢固定30%')

股票       环境     MA多头%     固定30%      V3仓位     胜者
-------------------------------------------------------
海康威视   牛市       59%     +3.2%     +4.3% (40%)    V3✅
恒瑞医药   熊市       17%     -3.4%     -2.3% (20%)    V3✅
比亚迪    熊市       26%     -3.1%     -2.0% (20%)    V3✅
茅台     震荡       31%     -4.3%     -4.3% (30%)      平
招商银行   熊市       29%     -1.4%     -1.0% (20%)    V3✅

📊 V3分级仓位在4/5股票上跑赢固定30%


## 五、今日总结

1. **凯利公式**：f*=(p×b-q)/b，只有海康的MACD策略有正凯利(23.7%)
2. **三种方法**：固定比例最简单，ATR法最科学（高波动自动减仓）
3. **连续亏损**：50%仓位连亏5次→资金跌超10%，回本需要涨更多
4. **三条铁律**：单笔≤5%风险、总仓≤60%、连亏3笔减半
5. **V3仓位**：牛市40%/震荡30%/熊市20%，4/5股票跑赢固定仓位

**核心认知**：仓位管理不是锦上添花，是生死线。跌50%需要涨100%才能回本。